# In Silico `Toxicity` prediction for `Drugbank + Drugcentral` FDA approved drug Candidates

This workflow uses `admet_ai` predictions to perform a **rule-based safety triage** of candidate molecules.

The goal is to support **early prioritization**, not to make definitive toxicity claims.

We separate predictions into three categories:

- **Core toxicity risk**: direct safety concerns such as hERG, DILI, AMES, and ClinTox
- **Mechanistic alerts**: stress-response and nuclear-receptor pathway signals
- **ADME / metabolism liabilities**: permeability, transporter, and CYP-related liabilities that may affect developability but do not necessarily imply intrinsic toxicity

**Approved Drugs Data has been downloaded from Drugbank (https://go.drugbank.com/releases/latest#open-data). Now we are going to convert it to csv file to run DORAnet for all the possible substructures.**

In [30]:
import sys
import os

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

In [31]:
import numpy as np
import pandas as pd
import pubchempy as pcp
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from admet_ai import ADMETModel

In [32]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsDir = os.path.join(dataDir, 'Toxicity/DrugBank/')
os.makedirs(resultsDir, exist_ok=True)

## Read `Drugbank` data

In [33]:
drugBankDrugDataDF_wSMILES = pd.read_csv(resultsDir + "drugBankDrugDataDF_wSMILES.csv", dtype=str)
print(f"Loaded {len(drugBankDrugDataDF_wSMILES)} molecules")
drugBankDrugDataDF_wSMILES

Loaded 14618 molecules


,DrugBank ID,Accession Numbers,Common name,CAS,UNII,Synonyms,Standard InChI Key,Canonical_SMILES
0,DB00006,BTD00076 | EXPT03302 | BIOD00076 | DB02351,Bivalirudin,128270-60-0,TN9BEX005G,Bivalirudin | Bivalirudina | Bivalirudinum,OIRCOABEOLEUMC-GEJPAHFPSA-N,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...
1,DB00014,BTD00113 | BIOD00113,Goserelin,65807-02-5,0F65R8P09N,Goserelin | Goserelina,BLCLNMBMMGCOAS-URPVMXJPSA-N,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2,DB00027,BTD00036 | BIOD00036,Gramicidin D,1405-97-6,5IE62321P4,Bacillus brevis gramicidin D | Gramicidin | Gr...,NDAYQJDHGXTBJL-MWWSRJDJSA-N,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3,DB00035,BTD00112 | BTD00061 | BIOD00112 | BIOD00061,Desmopressin,16679-58-6,ENR1LLB0FP,1 deamino 8 D argininevasopressin | 1-(3-merca...,NFLWUMRGJYTJIN-PNIOQBSNSA-N,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...
4,DB00050,BTD00115 | APRD00686 | BIOD00115,Cetrorelix,120287-85-6,OON1HFZ4BA,Cetrorelix | Cetrorelixum,SBNPWPIBESPSIF-MHWMIDJBSA-N,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...
...,...,...,...,...,...,...,...,...
14613,DB30552,NaN,Dipalmitoyl hydroxyproline,41672-81-5,E6AHA53N1H,"Dipalmitoyl hydroxyproline | O,n-dipalmitoylhy...",QZLXCFQVOCEKSX-NOCHOARKSA-N,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...
14614,DB31698,NaN,R-(-)-Gossypol acetic acid,866541-93-7,U9GNI6VT5N,(-)-gossypol acetic acid | (r)-gossypol acetic...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...
14615,DB31699,NaN,Gossypol acetic acid,5453-04-3,S7RL72610R,(+/-)-gossypol acetic acid | (±)-gossypol acet...,NIOHNDKHQHVLKA-UHFFFAOYSA-N,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...
14616,DB31734,NaN,FRAX597,1286739-19-2,NaN,NaN,DHUJCQOUWQMVCG-UHFFFAOYSA-N,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...


### Step 1 — Read predicted ADMET properties for all candidate SMILES

In [34]:
drugBankDrugDataDF_wSMILES_wToxicity = pd.read_csv(resultsDir + 'drugBankDrugDataDF_wSMILES_wToxicity.csv')
drugBankDrugDataDF_wSMILES_wToxicity

,Canonical_SMILES,molecular_weight,logP,hydrogen_bond_acceptors,hydrogen_bond_donors,Lipinski,QED,stereo_centers,tpsa,AMES,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@...,2180.317,-8.11643,29,28,1.0,0.014176,16,901.57,0.310810,...,1.046917,25.552540,59.053897,17.487398,16.130283,74.873982,20.666925,33.423808,48.623497,14.811943
1,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,1269.433,-3.10570,16,17,1.0,0.010267,9,495.89,0.576287,...,5.467235,42.303218,82.667701,1.706088,8.103916,40.558356,40.442032,50.368360,40.597131,5.622334
2,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,1811.253,4.86760,16,20,1.0,0.022619,13,519.89,0.278276,...,10.818147,67.235363,97.208220,97.983715,13.687476,94.416440,91.624661,93.563397,18.301667,53.431563
3,N=C(N)NCCC[C@@H](NC(=O)[C@@H]1CCCN1C(=O)[C@@H]...,1069.238,-4.13203,15,14,1.0,0.027501,7,435.41,0.230986,...,0.969368,49.515316,70.104692,18.883288,23.109732,55.874370,23.419930,34.160527,38.309422,20.473052
4,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,1431.064,-0.50613,16,17,1.0,0.013447,10,495.67,0.276726,...,3.916247,29.158589,75.261729,31.717720,10.585498,74.369911,44.358278,54.905002,45.017449,38.309422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14123,CCCCCCCCCCCCCCCC(=O)O[C@@H]1C[C@@H](C(=O)O)N(C...,607.961,10.54630,4,1,2.0,0.064844,2,83.91,0.023694,...,18.611865,91.081815,96.200078,17.564948,89.220628,9.112059,85.498255,98.526561,2.442807,29.895308
14124,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,578.614,6.47314,9,7,1.0,0.101848,0,192.82,0.100066,...,15.160915,6.165180,42.846064,95.851105,50.794882,61.535479,44.241954,83.210547,34.276851,58.045754
14125,CC(=O)O.Cc1cc2c(C(C)C)c(O)c(O)c(C=O)c2c(O)c1-c...,578.614,6.47314,9,7,1.0,0.101848,0,192.82,0.100066,...,15.160915,6.165180,42.846064,95.851105,50.794882,61.535479,44.241954,83.210547,34.276851,58.045754
14126,CCn1c(=O)c(-c2ccc(-c3cncs3)cc2Cl)cc2cnc(Nc3ccc...,558.111,5.75070,9,1,2.0,0.284585,0,79.18,0.280620,...,55.486623,30.632028,61.690578,90.732842,37.146181,49.554091,90.073672,92.943001,13.299729,92.051183


### Step 2 — retain potency predictions along with toxicity, mechanistic, and ADME-related endpoints

In [35]:
# Core toxicity endpoints
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu"
]

# Mechanistic alert endpoints
mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma"
]

# Mechanistic alert endpoints
# ADME / metabolism liability endpoints
admeLiabilityCols = [
    "BBB_Martins",
    "Pgp_Broccatelli",
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith"
]

# Keep only columns that exist
selectedCols = [
    col for col in (coreToxicityCols + mechanisticAlertCols + admeLiabilityCols)
    if col in drugBankDrugDataDF_wSMILES_wToxicity.columns
]

drugBankDrugDataDF_wSMILES_wToxicity = drugBankDrugDataDF_wSMILES_wToxicity[selectedCols].copy()

print("Filtered dataframe shape:", drugBankDrugDataDF_wSMILES_wToxicity.shape)
drugBankDrugDataDF_wSMILES_wToxicity.head()

Filtered dataframe shape: (14128, 33)


,AMES,hERG,DILI,ClinTox,Carcinogens_Lagunin,Skin_Reaction,LD50_Zhu,SR-ARE,SR-ATAD5,SR-HSE,...,PAMPA_NCATS,Bioavailability_Ma,CYP1A2_Veith,CYP2C19_Veith,CYP2C9_Substrate_CarbonMangels,CYP2C9_Veith,CYP2D6_Substrate_CarbonMangels,CYP2D6_Veith,CYP3A4_Substrate_CarbonMangels,CYP3A4_Veith
0,0.310810,0.203566,0.157990,0.117576,0.034491,0.183789,2.920528,0.097105,0.012814,0.006994,...,0.048505,0.257911,0.002577,0.022560,0.034454,0.003888,0.132853,0.180797,0.688971,0.086533
1,0.576287,0.761116,0.456281,0.369672,0.105431,0.098950,2.379779,0.064116,0.011034,0.008190,...,0.204220,0.378168,0.002031,0.054065,0.104391,0.054485,0.114031,0.145560,0.785328,0.423962
2,0.278276,0.818347,0.693615,0.084931,0.165993,0.073799,3.621140,0.335629,0.100212,0.068362,...,0.547499,0.439308,0.014953,0.290569,0.132428,0.223583,0.119283,0.031647,0.912250,0.933975
3,0.230986,0.228548,0.156015,0.054815,0.038481,0.154445,2.615314,0.106855,0.023336,0.012769,...,0.064727,0.227104,0.001188,0.014988,0.032820,0.002304,0.158653,0.082915,0.734302,0.149757
4,0.276726,0.769771,0.381344,0.164455,0.076909,0.082039,2.912322,0.103015,0.011854,0.012883,...,0.335595,0.400199,0.015246,0.185183,0.095395,0.076290,0.287348,0.330330,0.854831,0.771550


### Print range of all the endpoints

In [42]:
summaryStats = drugBankDrugDataDF_wSMILES_wToxicity[selectedCols].agg(
    ["min", "max", "mean", "std"]
).T

summaryStats.columns = ["Min", "Max", "Mean", "Std"]
summaryStats["Range"] = summaryStats["Max"] - summaryStats["Min"]
summaryStats = summaryStats[["Min", "Max", "Range", "Mean", "Std"]]

summaryStats.index.name = "Endpoint"

print("Summary statistics for all endpoints:")
print(summaryStats.to_string())

summaryStats.to_csv(resultsDir + 'drugBankDrugData_ADMET_summary.csv', index=True, encoding="utf-8")

Summary statistics for all endpoints:
                                         Min       Max     Range      Mean       Std
Endpoint                                                                            
AMES                            4.580234e-05  0.999994  0.999949  0.299619  0.249049
hERG                            5.135011e-05  0.997788  0.997737  0.434110  0.340795
DILI                            1.172916e-05  0.998585  0.998573  0.565484  0.337805
ClinTox                         6.131192e-09  0.955574  0.955574  0.217150  0.212429
Carcinogens_Lagunin             3.302558e-04  0.991252  0.990921  0.230227  0.199003
Skin_Reaction                   6.183962e-03  0.999993  0.993809  0.433260  0.253876
LD50_Zhu                       -4.707379e-02  6.023548  6.070622  2.546885  0.656044
SR-ARE                          2.767653e-05  0.999029  0.999002  0.272911  0.251478
SR-ATAD5                        3.475998e-13  0.986187  0.986187  0.058122  0.110646
SR-HSE                     

## Read the data of all the approved drugs from `Drugbank`

In [37]:
drugBankDrugDataDF_approved = pd.read_csv(resultsDir + 'drugbank_approved_structure_links.csv.zip')
drugBankDrugDataDF_approved.to_csv(resultsDir + 'drugbank_approved_structure_links.csv', index=False, encoding="utf-8")
drugBankDrugDataDF_approved

,DrugBank ID,Name,CAS Number,Drug Groups,InChIKey,InChI,SMILES,Formula,KEGG Compound ID,KEGG Drug ID,PubChem Compound ID,PubChem Substance ID,ChEBI ID,ChEMBL ID,HET ID,ChemSpider ID,BindingDB ID
0,DB00006,Bivalirudin,128270-60-0,approved; investigational,OIRCOABEOLEUMC-GEJPAHFPSA-N,InChI=1S/C98H138N24O33/c1-5-52(4)82(96(153)122...,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,C98H138N24O33,NaN,D03136,16129704.0,46507415.0,59173.0,CHEMBL2103749,NaN,10482069.0,50248103.0
1,DB00014,Goserelin,65807-02-5,approved; investigational,BLCLNMBMMGCOAS-URPVMXJPSA-N,InChI=1S/C59H84N18O14/c1-31(2)22-40(49(82)68-3...,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,C59H84N18O14,NaN,D00573,5311128.0,46507336.0,5523.0,CHEMBL1201247,NaN,4470656.0,50247974.0
2,DB00027,Gramicidin D,1405-97-6,approved; investigational,NDAYQJDHGXTBJL-MWWSRJDJSA-N,InChI=1S/C96H135N19O16/c1-50(2)36-71(105-79(11...,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,C96H135N19O16,NaN,D04369,45267103.0,46507412.0,NaN,CHEMBL557217,NaN,24623445.0,NaN
3,DB00035,Desmopressin,16679-58-6,approved; investigational,NFLWUMRGJYTJIN-PNIOQBSNSA-N,InChI=1S/C46H64N14O12S2/c47-35(62)15-14-29-40(...,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,C46H64N14O12S2,C06944,D00291,NaN,NaN,4450.0,CHEMBL1429,NaN,4470602.0,50247923.0
4,DB00050,Cetrorelix,120287-85-6,approved; investigational,SBNPWPIBESPSIF-MHWMIDJBSA-N,InChI=1S/C70H92ClN17O14/c1-39(2)31-52(61(94)82...,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...,C70H92ClN17O14,NaN,D07665,25074887.0,46505494.0,59224.0,CHEMBL1200490,NaN,10482082.0,50369965.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3015,DB20138,Tribromsalan,87-10-5,approved; withdrawn,KVSKGMLNBAPGKH-UHFFFAOYSA-N,InChI=1S/C13H8Br3NO2/c14-7-1-3-9(4-2-7)17-13(1...,OC1=C(C=C(Br)C=C1Br)C(=O)NC1=CC=C(Br)C=C1,C13H8Br3NO2,NaN,NaN,NaN,NaN,127105.0,CHEMBL24944,NaN,NaN,234329.0
3016,DB20831,Iopydone,5579-93-1,approved; withdrawn,FRPFEVLOFNAKBS-UHFFFAOYSA-N,"InChI=1S/C5H3I2NO/c6-3-1-8-2-4(7)5(3)9/h1-2H,(...",IC1=CNC=C(I)C1=O,C5H3I2NO,NaN,NaN,NaN,NaN,NaN,CHEMBL2104367,NaN,NaN,NaN
3017,DB21667,Sevabertinib,2521285-05-0,approved; investigational,VYQVHWNNPKOJEA-AWEZNQCLSA-N,InChI=1S/C24H25ClN4O5/c1-31-23-16(25)3-2-4-18(...,COC1=C(NC2=C(NC3=C2C(=O)NCC3)C2=CC=NC=C2OC[C@@...,C24H25ClN4O5,NaN,NaN,NaN,NaN,NaN,CHEMBL6068074,NaN,NaN,NaN
3018,DB21811,Gozetotide,1366302-52-4,approved; investigational,QJUIUFGOTBRHKP-LQJZCPKCSA-N,InChI=1S/C44H62N6O17/c51-34-13-8-28(22-30(34)2...,OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=O)CCCCCNC(...,C44H62N6O17,NaN,NaN,NaN,NaN,NaN,CHEMBL3578202,NaN,NaN,50089451.0


### Step 1 — Read predicted ADMET properties for all candidate SMILES

In [38]:
drugBankDrugDataDF_approved_wToxicity = pd.read_csv(resultsDir + 'drugbank_approved_structure_links_wToxicity.csv')
drugBankDrugDataDF_approved_wToxicity

,SMILES,molecular_weight,logP,hydrogen_bond_acceptors,hydrogen_bond_donors,Lipinski,QED,stereo_centers,tpsa,AMES,...,Caco2_Wang_drugbank_approved_percentile,Clearance_Hepatocyte_AZ_drugbank_approved_percentile,Clearance_Microsome_AZ_drugbank_approved_percentile,Half_Life_Obach_drugbank_approved_percentile,HydrationFreeEnergy_FreeSolv_drugbank_approved_percentile,LD50_Zhu_drugbank_approved_percentile,Lipophilicity_AstraZeneca_drugbank_approved_percentile,PPBR_AZ_drugbank_approved_percentile,Solubility_AqSolDB_drugbank_approved_percentile,VDss_Lombardo_drugbank_approved_percentile
0,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,2180.317,-8.11643,29,28,1.0,0.014176,16,901.57,0.310810,...,1.046917,25.591314,59.053897,17.487398,16.169058,74.873982,20.666925,33.462582,48.623497,14.811943
1,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,1269.433,-3.10570,16,17,1.0,0.010267,9,495.89,0.576287,...,5.467235,42.303218,82.667701,1.706088,8.103916,40.558356,40.442032,50.368360,40.597131,5.622334
2,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,1811.253,4.86760,16,20,1.0,0.022619,13,519.89,0.279331,...,10.856921,67.390461,97.246995,98.022489,13.842575,94.377666,91.973633,93.834820,18.069019,54.013183
3,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,1069.238,-4.13203,15,14,1.0,0.027501,7,435.41,0.246513,...,1.202016,51.221404,73.129120,18.069019,24.428073,59.015122,25.358666,36.952307,36.603335,22.217914
4,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...,1431.064,-0.50613,16,17,1.0,0.013447,10,495.67,0.278756,...,3.916247,28.770841,74.718883,31.950368,10.507949,73.904614,43.660333,54.284606,45.405196,37.495153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2773,OC1=C(C=C(Br)C=C1Br)C(=O)NC1=CC=C(Br)C=C1,449.924,4.93200,2,2,4.0,0.682461,0,49.33,0.139711,...,55.796820,84.296239,71.074060,82.900349,48.545948,53.702986,94.920512,97.789841,7.444746,6.630477
2774,IC1=CNC=C(I)C1=O,346.893,1.58410,1,1,4.0,0.710881,0,32.86,0.091733,...,84.645211,57.115161,52.035673,79.139201,32.493214,46.994959,44.397053,39.511439,57.115161,96.161303
2775,COC1=C(NC2=C(NC3=C2C(=O)NCC3)C2=CC=NC=C2OC[C@@...,484.940,3.56240,7,3,4.0,0.470385,1,106.73,0.424901,...,31.678945,54.749903,65.451725,95.075611,20.279178,80.884064,81.155487,88.794106,11.593641,90.538969
2776,OC(=O)CC[C@H](NC(=O)N[C@@H](CCCCNC(=O)CCCCCNC(...,947.005,1.55450,13,12,1.0,0.043615,2,370.07,0.364973,...,4.071345,23.613804,59.790617,83.637069,18.456766,31.640171,16.130283,50.019387,78.402482,26.832105


### Step 2 — retain potency predictions along with toxicity, mechanistic, and ADME-related endpoints

In [39]:
# Core toxicity endpoints
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu"
]

# Mechanistic alert endpoints
mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma"
]

# Mechanistic alert endpoints
# ADME / metabolism liability endpoints
admeLiabilityCols = [
    "BBB_Martins",
    "Pgp_Broccatelli",
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith"
]

# Keep only columns that exist
selectedCols = [
    col for col in (coreToxicityCols + mechanisticAlertCols + admeLiabilityCols)
    if col in drugBankDrugDataDF_approved_wToxicity.columns
]

drugBankDrugDataDF_approved_wToxicity = drugBankDrugDataDF_approved_wToxicity[selectedCols].copy()

print("Filtered dataframe shape:", drugBankDrugDataDF_approved_wToxicity.shape)
drugBankDrugDataDF_approved_wToxicity.head()

Filtered dataframe shape: (2778, 33)


,AMES,hERG,DILI,ClinTox,Carcinogens_Lagunin,Skin_Reaction,LD50_Zhu,SR-ARE,SR-ATAD5,SR-HSE,...,PAMPA_NCATS,Bioavailability_Ma,CYP1A2_Veith,CYP2C19_Veith,CYP2C9_Substrate_CarbonMangels,CYP2C9_Veith,CYP2D6_Substrate_CarbonMangels,CYP2D6_Veith,CYP3A4_Substrate_CarbonMangels,CYP3A4_Veith
0,0.310810,0.203566,0.157990,0.117576,0.034491,0.183789,2.920528,0.097105,0.012814,0.006994,...,0.048505,0.257911,0.002577,0.022560,0.034454,0.003888,0.132853,0.180797,0.688971,0.086533
1,0.576287,0.761116,0.456281,0.369672,0.105431,0.098950,2.379779,0.064116,0.011034,0.008190,...,0.204220,0.378168,0.002031,0.054065,0.104391,0.054485,0.114031,0.145560,0.785328,0.423962
2,0.279331,0.820039,0.693912,0.084993,0.163415,0.074538,3.619984,0.340856,0.100954,0.069387,...,0.551455,0.436321,0.015546,0.298653,0.131439,0.232136,0.119051,0.033491,0.911799,0.936834
3,0.246513,0.253157,0.167744,0.058355,0.039024,0.149433,2.658192,0.133861,0.037735,0.017046,...,0.077015,0.216245,0.001653,0.021896,0.036195,0.003967,0.164736,0.095306,0.745049,0.218569
4,0.278756,0.763266,0.376340,0.169496,0.078289,0.081460,2.904686,0.099557,0.011483,0.012237,...,0.333679,0.404840,0.014637,0.170699,0.094969,0.068207,0.294748,0.321523,0.854312,0.765157


### Print range of all the endpoints

In [40]:
summaryStats = drugBankDrugDataDF_approved_wToxicity[selectedCols].agg(
    ["min", "max", "mean", "std"]
).T

summaryStats.columns = ["Min", "Max", "Mean", "Std"]
summaryStats["Range"] = summaryStats["Max"] - summaryStats["Min"]

summaryStats = summaryStats[["Min", "Max", "Range", "Mean", "Std"]]

print("Summary statistics for all endpoints:")
print(summaryStats.to_string())
summaryStats.to_csv(resultsDir + 'drugBankDrugData_approved_ADMET_summary.csv', index=False, encoding="utf-8")

Summary statistics for all endpoints:
                                         Min       Max     Range      Mean       Std
AMES                            5.837759e-05  0.999994  0.999936  0.267404  0.246170
hERG                            1.022494e-04  0.996943  0.996841  0.396913  0.346444
DILI                            1.754974e-05  0.997723  0.997706  0.486358  0.347489
ClinTox                         1.182336e-05  0.904089  0.904077  0.187615  0.206918
Carcinogens_Lagunin             3.267163e-04  0.989182  0.988855  0.226055  0.206324
Skin_Reaction                   1.151846e-02  0.998180  0.986662  0.463766  0.269045
LD50_Zhu                        2.533023e-01  5.018555  4.765253  2.527885  0.671969
SR-ARE                          8.631757e-05  0.994185  0.994098  0.232714  0.247490
SR-ATAD5                        7.235494e-08  0.934061  0.934061  0.043791  0.099153
SR-HSE                          5.968973e-07  0.959575  0.959575  0.062054  0.132050
SR-MMP                     